In [1]:
import numpy as np

In [ ]:
# function for generating immigrants and descendants
# code taken from https://arxiv.org/pdf/2502.18979 Algorithm 1

# mu - d dimensional vector
# alpha - dxd dimensional matrix
# beta - scalar > 0
# time - scalar > 0
def simulation_by_cluster_representation(mu, alpha, beta, time):
    d = len(mu) # dimension

    # family tree is not part of the paper's algorithm
    family_trees = [{} for _ in range(d)] # timestamp of event: (parent variable, timestamp of parent). parents of immigrants are -1

    # initialization
    T = [[] for _ in range(d)] # list of all immigrants and descendants
    A = [[] for _ in range(d)] # temporary list of ancestors

    # immigrant simulation
    for j in range(d):
        k = np.random.poisson(lam=mu[j]*time) # number of immigrants of type j

        small_t = [[] for _ in range(d)] # t != T (from the paper's algorithm)
        for _ in range(k):
            small_t[j].append(np.random.uniform(low=0, high=time)) # small t is different from big T

        A[j] = small_t[j]
        T[j] = list(set(T[j] + A[j]))

        for immigrant in T[j]: # assigns the parent of each immigrant to -1
            family_trees[j][immigrant] = (j, -1)

    # helper function for while loop condition (self explanatory)
    def there_exists_at_least_one_j_st_Aj_neq_empty(A):
        for j in range(len(A)):
            if A[j] != []:
                return True
        return False

    # offspring simulation
    while there_exists_at_least_one_j_st_Aj_neq_empty(A):
        O = [[] for _ in range(d)] # offspring initialization
        for j in range(d):
            if A[j] != []:
                for l in range(len(A[j])):
                    for j_prime in range(d):
                        if alpha[j_prime][j] > 0:
                            k = np.random.poisson(lam=alpha[j_prime][j])
                            
                            # finds elapsed time for descendants
                            small_t = [[] for _ in range(d)]
                            for i in range(k):
                                small_t[j_prime].append(np.random.exponential(beta))

                            # adds elapsed time of descendant to the timestamp of its parent (A[j][l] is the parent)
                            a_plus_t = []
                            for i in range(k):
                                a_plus_t.append(A[j][l] + small_t[j_prime][i])
                            O[j_prime] = list(set(O[j_prime] + a_plus_t))

                            # assigns the parent of each descendant
                            for descendant in a_plus_t:
                                family_trees[j_prime][descendant] = (j, A[j][l])

                        T[j_prime] = list(set(T[j_prime] + O[j_prime])) # offsprings are added to T
        
        A = O # offspring become ancestors

    # remove events beyond T and sort
    for j in range(d):
        i = 0
        while i < len(T[j]): # removes events that occur after time limit
            if T[j][i] > time:
                T[j].pop(i)
                i -= 1
            i += 1
        T[j] = sorted(T[j])

        # remove events from family_tree if they're beyond T
        for key in list(family_trees[j].keys()):
            if key > time:
                family_trees[j].pop(key)
    
    return T, family_trees

In [ ]:
# generate data

# model parameters
mu = [0.1, 0.1] # background intensity
alpha = [[0.1, 0.4],
         [0.4, 0.1]] # mutual excitation matrix; alpha_12 means that 1 is excited by 2
beta = 1 # decay
time = 300 # time

np.random.seed(0)
timestamps, family_trees = simulation_by_cluster_representation(mu, alpha, beta, time)

for i in timestamps:
    print(len(i), i)

for tree in family_trees:
    print()
    for key in sorted(tree.keys()):
        print(key, ':', tree[key])

68 [6.065519232097715, 21.31081745936608, 26.13878991046221, 31.38256303074278, 33.3524658718764, 33.842770019892804, 35.482327760679965, 43.00598622271392, 48.2487536175206, 63.62263343624833, 63.75093126068636, 79.36668363138809, 111.34607503275501, 111.75880489314568, 115.03245564773331, 115.61623769769473, 117.33530701473684, 117.6533960218519, 124.39858199715708, 127.09643980167141, 130.85836697379094, 131.27616337880775, 136.84509966496455, 137.03529179855943, 137.3558951854002, 137.47367996928247, 138.33735201305552, 138.44380867587955, 139.91003952774778, 156.5544965250215, 158.66847592587135, 160.10272518286502, 163.46495489906906, 164.38053491826824, 165.22087574388948, 170.4133683281797, 180.82901282149317, 185.4679747018743, 185.60306909175478, 185.82954680744595, 187.3165579155051, 188.70246807294814, 189.75844358135998, 191.97630639825715, 193.76823391999685, 201.95844055238302, 204.99234256000824, 205.8553880976547, 209.71301588281244, 210.70847952975365, 212.80583136702